# 01. 중분류 그룹 접근성-이용 상관분석
- 목적: 음악·체육용품 중분류와 그외 8개 중분류의 접근성-이용건수 상관 패턴을 비교함.
- 비교 지표: 선호 미반영 2SFCA, 최근접 접근성(-거리), 도달권역 인구비중, 종합문화취약지수.
- 이용 지표: 2025년 문화누리대상자 추정인구 1인당 중분류 이용건수.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

U = lambda s: s.encode('ascii').decode('unicode_escape')
PROJECT = Path(r'C:\project\oracle_mnc_project')
EDA_OUTPUT = PROJECT / 'notebooks' / 'eda' / 'OUTPUT'
H3 = PROJECT / 'notebooks' / 'access' / 'OUTPUT' / 'h3sfca'
PUBLIC = PROJECT / 'notebooks' / 'access' / 'OUTPUT' / 'public_access_index_25km'
VULN = PROJECT / 'notebooks' / 'access' / 'OUTPUT' / 'final_vulnerability_index'
MNC = PROJECT / 'data' / 'raw' / 'mnc_card' / 'mnc_seoul_usage_issuance_2021_2025.xlsx'
EDA_OUTPUT.mkdir(parents=True, exist_ok=True)

GRID = 'GRID_CD'
GU = U('\\uc2dc\\uad70\\uad6c')
MID = U('\\uc911\\ubd84\\ub958')
GWANG = U('\\uad11\\uc5ed')
GICHO = U('\\uae30\\ucd08')
SEOUL = U('\\uc11c\\uc6b8')
MNC_POP = U('\\ubb38\\ud654\\ub204\\ub9ac\\ub300\\uc0c1\\uc790_\\ucd94\\uc815_\\uc778\\uad6c\\uc218')
ACCESS = U('\\uc811\\uadfc\\uc131\\uc9c0\\uc218')
DIST = U('\\ubb38\\ud654\\ub204\\ub9ac\\ub300\\uc0c1\\uc790_\\uac00\\uc911\\ud3c9\\uade0_\\uc811\\uadfc\\uac70\\ub9ac_m')
SERVICE_RATIO = U('\\ubb38\\ud654\\ub204\\ub9ac\\ub300\\uc0c1\\uc790_\\uc11c\\ube44\\uc2a4\\uad8c\\uc5ed\\ube44\\uc728')
ANALYSIS_TARGET = U('\\ubd84\\uc11d\\ub300\\uc0c1')
FINAL_VULN = U('\\ucd5c\\uc885\\ucde8\\uc57d\\uc9c0\\uc218_z')
USE_COUNT = U('\\uc774\\uc6a9\\uac74\\uc218')
GU_MNC_POP = U('\\uad6c\\ubcc4_\\ubb38\\ud654\\ub204\\ub9ac\\ub300\\uc0c1\\uc790\\ucd94\\uc815\\uc778\\uad6c')
PER_CAPITA = U('\\ub300\\uc0c1\\uc7901\\uc778\\ub2f9_\\uc774\\uc6a9\\uac74\\uc218')
PER_1000 = U('\\ub300\\uc0c1\\uc790\\ucc9c\\uba85\\ub2f9_\\uc774\\uc6a9\\uac74\\uc218')
GROUP = U('\\ubd84\\ub958\\uadf8\\ub8f9')
IND = U('\\uc9c0\\ud45c\\uba85')
VALUE = U('\\uc9c0\\ud45c\\uac12')
VALUE_MEAN = U('\\uc9c0\\ud45c\\uac12_\\ud3c9\\uade0')
USE_MEAN = U('\\ub300\\uc0c1\\uc790\\ucc9c\\uba85\\ub2f9_\\uc774\\uc6a9\\uac74\\uc218_\\ud3c9\\uade0')
CAT_N = U('\\uc911\\ubd84\\ub958\\uc218')
PEARSON_MEAN = 'Pearson_' + U('\\ud3c9\\uade0')
PEARSON_MEDIAN = 'Pearson_' + U('\\uc911\\uc559\\uac12')
PEARSON_POS = 'Pearson_' + U('\\uc591\\uc218\\ubd84\\ub958\\uc218')
SPEARMAN_MEAN = 'Spearman_' + U('\\ud3c9\\uade0')
SPEARMAN_MEDIAN = 'Spearman_' + U('\\uc911\\uc559\\uac12')
SPEARMAN_POS = 'Spearman_' + U('\\uc591\\uc218\\ubd84\\ub958\\uc218')
IND_SFCA = '2SFCA_' + U('\\uc120\\ud638\\ubbf8\\ubc18\\uc601')
IND_NEAREST = U('\\ucd5c\\uadfc\\uc811\\uc811\\uadfc\\uc131_\\uc74c\\uac70\\ub9ac')
IND_SERVICE = U('\\ub3c4\\ub2ec\\uad8c\\uc5ed_\\uc778\\uad6c\\ube44\\uc911')
IND_VULN = U('\\uc885\\ud569\\ubb38\\ud654\\ucde8\\uc57d\\uc9c0\\uc218_z')
GROUP_TARGET = U('\\uc74c\\uc545\\u00b7\\uccb4\\uc721\\uc6a9\\ud488')
GROUP_OTHER = U('\\uadf8\\uc678_8\\uac1c\\uc911\\ubd84\\ub958')

cat = {
    U('\\ub3c4\\uc11c'): [U('\\ub3c4\\uc11c')],
    U('\\uc74c\\uc545'): [U('\\uc74c\\uc545')],
    U('\\uc601\\uc0c1'): [U('\\uc601\\ud654'), 'TV'],
    U('\\uacf5\\uc5f0'): [U('\\uacf5\\uc5f0')],
    U('\\ubbf8\\uc220'): [U('\\uc804\\uc2dc'), U('\\uacf5\\uc608'), U('\\uc0ac\\uc9c4\\uad00')],
    U('\\ubb38\\ud654\\uccb4\\ud5d8'): [U('\\ubb38\\ud654\\uccb4\\ud5d8'), U('\\uc9c1\\uc5c5\\uccb4\\ud5d8'), U('\\ubb38\\ud654\\uc77c\\ubc18')],
    U('\\uad00\\uad11\\uc9c0'): [U('\\uad00\\uad11\\uba85\\uc18c'), U('\\ud734\\uc591\\ub9bc\\ucea0\\ud551\\uc7a5'), U('\\ub3d9\\uc2dd\\ubb3c\\uc6d0'), U('\\uc628\\ucc9c'), U('\\uccb4\\ud5d8\\uad00\\uad11'), U('\\ud14c\\ub9c8\\ud30c\\ud06c')],
    U('\\uc2a4\\ud3ec\\uce20\\uad00\\ub78c'): [U('\\uc2a4\\ud3ec\\uce20\\uad00\\ub78c')],
    U('\\uccb4\\uc721\\uc6a9\\ud488'): [U('\\uccb4\\uc721\\uc6a9\\ud488')],
    U('\\uccb4\\uc721\\uc2dc\\uc124'): [U('\\uccb4\\uc721\\uc2dc\\uc124')],
}
target_group = [U('\\uc74c\\uc545'), U('\\uccb4\\uc721\\uc6a9\\ud488')]
cat_list = list(cat.keys())

def norm(v):
    v = str(v).strip()
    for t in ['\n', ' ', '/', U('\\u318d'), U('\\u00b7'), '(', ')']:
        v = v.replace(t, '')
    return v

def clean_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '', regex=False).str.strip(), errors='coerce')

def weighted_mean(value, weight):
    value = pd.to_numeric(value, errors='coerce')
    weight = pd.to_numeric(weight, errors='coerce').fillna(0)
    valid = value.notna()
    if valid.sum() == 0:
        return np.nan
    value = value[valid]
    weight = weight[valid]
    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()

def safe_corr(df, x, y, method):
    temp = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) < 3:
        return np.nan
    if temp[x].nunique() < 2 or temp[y].nunique() < 2:
        return np.nan
    return temp[x].corr(temp[y], method=method)

sfca_grid_path = H3 / U('sfca_no_preference_\\uaca9\\uc790_\\uc911\\ubd84\\ub958_\\uc811\\uadfc\\uc131.csv')
base = pd.read_csv(sfca_grid_path, encoding='utf-8-sig', usecols=[GRID, GU, MNC_POP]).drop_duplicates(GRID)
base[MNC_POP] = pd.to_numeric(base[MNC_POP], errors='coerce').fillna(0)
gu_pop = base.groupby(GU, as_index=False).agg(**{GU_MNC_POP: (MNC_POP, 'sum')})

raw = pd.read_excel(MNC, sheet_name='2025')
raw[GWANG] = raw[GWANG].astype(str).str.strip()
raw[GICHO] = raw[GICHO].astype(str).str.strip()
raw = raw[raw[GWANG].eq(SEOUL)].rename(columns={GICHO: GU}).copy()
count_cols = [c for c in raw.columns if str(c).endswith(U('(\\uac74)'))]
lookup = {norm(str(c).replace(U('(\\uac74)'), '')): c for c in count_cols}
frames = []
missing = []
for mid, raw_names in cat.items():
    temp = raw[[GU]].copy()
    temp[USE_COUNT] = 0.0
    for name in raw_names:
        col = lookup.get(norm(name))
        if col is None:
            missing.append((mid, name))
            continue
        temp[USE_COUNT] = temp[USE_COUNT] + clean_num(raw[col]).fillna(0)
    temp[MID] = mid
    frames.append(temp)
usage = pd.concat(frames, ignore_index=True).merge(gu_pop, on=GU, how='left')
usage[PER_CAPITA] = np.where(usage[GU_MNC_POP] > 0, usage[USE_COUNT] / usage[GU_MNC_POP], np.nan)
usage[PER_1000] = usage[PER_CAPITA] * 1000
usage[GROUP] = np.where(usage[MID].isin(target_group), GROUP_TARGET, GROUP_OTHER)

sfca = pd.read_csv(sfca_grid_path, encoding='utf-8-sig', usecols=[GU, MID, ACCESS, MNC_POP])
sfca = sfca[sfca[MID].isin(cat_list)].copy()
sfca_gu = sfca.groupby([GU, MID], as_index=False).apply(lambda x: pd.Series({VALUE: weighted_mean(x[ACCESS], x[MNC_POP])}), include_groups=False)
sfca_gu[IND] = IND_SFCA
sfca_gu = sfca_gu[[GU, MID, IND, VALUE]]

nearest = pd.read_csv(PUBLIC / U('\\uacf5\\uacf5\\uae30\\uad00\\uc2dd_\\ucd5c\\uadfc\\uc811\\uc811\\uadfc\\uc131_\\uc11c\\uc6b8\\uc2dc\\uad70\\uad6c_\\uc911\\ubd84\\ub958\\ubcc4.csv'), encoding='utf-8-sig')
nearest = nearest[nearest[MID].isin(cat_list)].copy()
nearest[DIST] = pd.to_numeric(nearest[DIST], errors='coerce')
nearest_gu = nearest[[GU, MID, DIST]].copy()
nearest_gu[IND] = IND_NEAREST
nearest_gu[VALUE] = -nearest_gu[DIST]
nearest_gu = nearest_gu[[GU, MID, IND, VALUE]]

service = pd.read_csv(PUBLIC / U('\\uacf5\\uacf5\\uae30\\uad00\\uc2dd_\\uc11c\\ube44\\uc2a4\\uad8c\\uc5ed\\uc778\\uad6c\\ube44\\uc728_\\uc11c\\uc6b8\\uc2dc\\uad70\\uad6c_\\uc911\\ubd84\\ub958\\ubcc4.csv'), encoding='utf-8-sig')
service = service[service[MID].isin(cat_list)].copy()
service_gu = service[[GU, MID, SERVICE_RATIO]].copy()
service_gu[IND] = IND_SERVICE
service_gu[VALUE] = pd.to_numeric(service_gu[SERVICE_RATIO], errors='coerce')
service_gu = service_gu[[GU, MID, IND, VALUE]]

vuln = pd.read_csv(VULN / U('\\uc885\\ud569\\ubb38\\ud654\\ucde8\\uc57d\\uc9c0\\uc218_\\uc120\\ud638\\ubbf8\\ubc18\\uc601_SFCA.csv'), encoding='utf-8-sig', usecols=[GRID, GU, MNC_POP, ANALYSIS_TARGET, FINAL_VULN])
vuln = vuln[vuln[ANALYSIS_TARGET].eq(True)].copy()
vuln[MNC_POP] = pd.to_numeric(vuln[MNC_POP], errors='coerce').fillna(0)
vuln[FINAL_VULN] = pd.to_numeric(vuln[FINAL_VULN], errors='coerce')
vuln_gu = vuln.groupby(GU, as_index=False).apply(lambda x: pd.Series({VALUE: weighted_mean(x[FINAL_VULN], x[MNC_POP])}), include_groups=False)
vuln_gu = vuln_gu.merge(pd.DataFrame({MID: cat_list}), how='cross')
vuln_gu[IND] = IND_VULN
vuln_gu = vuln_gu[[GU, MID, IND, VALUE]]

indicator = pd.concat([sfca_gu, nearest_gu, service_gu, vuln_gu], ignore_index=True)
analysis = indicator.merge(usage[[GU, MID, GROUP, USE_COUNT, GU_MNC_POP, PER_CAPITA, PER_1000]], on=[GU, MID], how='left')

cat_rows = []
for (grp, mid, ind), temp in analysis.groupby([GROUP, MID, IND]):
    cat_rows.append({GROUP: grp, MID: mid, IND: ind, 'n': temp[[VALUE, PER_CAPITA]].replace([np.inf, -np.inf], np.nan).dropna().shape[0], 'Pearson': safe_corr(temp, VALUE, PER_CAPITA, 'pearson'), 'Spearman': safe_corr(temp, VALUE, PER_CAPITA, 'spearman'), VALUE_MEAN: temp[VALUE].mean(), USE_MEAN: temp[PER_1000].mean()})
cat_corr = pd.DataFrame(cat_rows).sort_values([GROUP, MID, IND]).reset_index(drop=True)

summary = cat_corr.groupby([GROUP, IND], as_index=False).agg(**{
    CAT_N: (MID, 'nunique'),
    PEARSON_MEAN: ('Pearson', 'mean'),
    PEARSON_MEDIAN: ('Pearson', 'median'),
    PEARSON_POS: ('Pearson', lambda x: (x > 0).sum()),
    SPEARMAN_MEAN: ('Spearman', 'mean'),
    SPEARMAN_MEDIAN: ('Spearman', 'median'),
    SPEARMAN_POS: ('Spearman', lambda x: (x > 0).sum()),
})

pooled_rows = []
for (grp, ind), temp in analysis.groupby([GROUP, IND]):
    pooled_rows.append({GROUP: grp, IND: ind, 'n': temp[[VALUE, PER_CAPITA]].replace([np.inf, -np.inf], np.nan).dropna().shape[0], 'Pearson': safe_corr(temp, VALUE, PER_CAPITA, 'pearson'), 'Spearman': safe_corr(temp, VALUE, PER_CAPITA, 'spearman')})
pooled = pd.DataFrame(pooled_rows).sort_values([GROUP, IND]).reset_index(drop=True)

cat_corr.to_csv(EDA_OUTPUT / 'middle_category_access_usage_correlation_2025.csv', index=False, encoding='utf-8-sig')
summary.to_csv(EDA_OUTPUT / 'middle_category_group_correlation_summary_2025.csv', index=False, encoding='utf-8-sig')
pooled.to_csv(EDA_OUTPUT / 'middle_category_group_pooled_correlation_2025.csv', index=False, encoding='utf-8-sig')

print('missing source columns:', missing)
print('analysis rows:', analysis.shape)
display(summary.round(4))
display(cat_corr[cat_corr[GROUP].eq(GROUP_TARGET)].round(4))
display(pooled.round(4))


## 02. 결과 테이블
- 중분류별 상관계수와 그룹 요약을 별도로 저장함.
- 그룹 전체를 묶은 pooled 상관은 중분류별 규모 차이가 영향을 줄 수 있어 참고값으로만 사용함.

In [ ]:

from pathlib import Path
import pandas as pd

U = lambda s: s.encode('ascii').decode('unicode_escape')
PROJECT = Path(r'C:\project\oracle_mnc_project')
OUT = PROJECT / 'notebooks' / 'eda' / 'OUTPUT'
GROUP = U('\\ubd84\\ub958\\uadf8\\ub8f9')
GROUP_TARGET = U('\\uc74c\\uc545\\u00b7\\uccb4\\uc721\\uc6a9\\ud488')

summary = pd.read_csv(OUT / 'middle_category_group_correlation_summary_2025.csv', encoding='utf-8-sig')
category = pd.read_csv(OUT / 'middle_category_access_usage_correlation_2025.csv', encoding='utf-8-sig')
pooled = pd.read_csv(OUT / 'middle_category_group_pooled_correlation_2025.csv', encoding='utf-8-sig')

print('group summary')
display(summary.round(4))
print('music and sports goods category detail')
display(category[category[GROUP].eq(GROUP_TARGET)].round(4))
print('pooled reference')
display(pooled.round(4))


## 03. 결과 확인
- 음악·체육용품 2개 중분류와 그외 8개 중분류의 Pearson/Spearman 상관을 비교함.
- 이용지표는 문화누리대상자 추정인구 1인당 2025년 이용건수임.
- 최근접 접근성은 방향을 맞추기 위해 거리의 음수값을 사용함.
- 종합문화취약지수는 접근성이 낮고 취약할수록 높은 값이라, 이용건수와의 방향성을 별도로 해석해야 함.


In [ ]:
# 결과 확인
summary_result = pd.read_csv(OUTPUT_PATH / 'middle_category_group_correlation_summary_2025.csv', encoding='utf-8-sig')
detail_result = pd.read_csv(OUTPUT_PATH / 'middle_category_access_usage_correlation_2025.csv', encoding='utf-8-sig')

print('그룹 요약')
display(summary_result)

print('음악·체육용품 상세')
display(detail_result[detail_result['분류그룹'] == '음악·체육용품'].sort_values(['중분류', '지표명']))


### 결과 시각화
- 그룹 요약 막대그래프와 중분류별 Pearson/Spearman 히트맵을 확인함.
- 이미지 저장 위치: `notebooks/eda/IMAGE`


In [ ]:
from IPython.display import Image, display

for image_name in [
    'middle_category_group_correlation_summary_table_2025.png',
    'middle_category_group_correlation_bar_2025.png',
    'middle_category_pearson_heatmap_2025.png',
    'middle_category_spearman_heatmap_2025.png',
]:
    print(image_name)
    display(Image(filename=str(IMAGE_PATH / image_name)))
